# ETL Cleaning — Order Payments Table
**Source:** `olist_order_payments_dataset.csv`  
**Output:** `data/cleaned/order_payments_cleaned.csv`

### Key characteristics of this table
- One order can have multiple payment rows (e.g. credit card + voucher), identified by `payment_sequential`
- `payment_sequential` is the sequence number of each payment method used within a single order
- `payment_installments` is the number of instalments for that specific payment method

### Cleaning Steps
1. Load raw data
2. Initial inspection (shape, columns, missing values, distributions)
3. Investigate anomalies (zero installments, zero payment value, not_defined type)
4. Remove invalid rows
5. Check composite primary key duplicates
6. Export cleaned data

## Step 1 — Load Raw Data

In [10]:
import pandas as pd

# ── 1. Load raw data ──────────────────────────────────────────
df = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

## Step 2 — Initial Inspection

In [11]:
# ── 2. Initial inspection ─────────────────────────────────────
print("=== Shape ===")
print(df.shape)

print("\n=== Column names ===")
print(df.columns.tolist())

print("\n=== First 5 rows ===")
print(df.head())

print("\n=== Current dtypes ===")
print(df.dtypes)

print("\n=== Missing values ===")
print(df.isnull().sum())

print("\n=== Numeric columns summary ===")
print(df.describe())

print("\n=== Categorical columns summary ===")
print(df.describe(include='str'))

=== Shape ===
(103886, 5)

=== Column names ===
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

=== First 5 rows ===
                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card   
4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card   

   payment_installments  payment_value  
0                     8          99.33  
1                     1          24.39  
2                     1          65.71  
3                     8         107.78  
4                     2         128.45  

=== Current dtypes ===
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment

## Step 3 — Investigate Anomalies

From the initial inspection, three anomalies were identified:
- `payment_installments` min = 0 (credit card with 0 instalments is invalid; 1 means pay in full)
- `payment_value` min = 0 (a payment row with zero value is invalid)
- `payment_type` has a `not_defined` category

We investigate each before deciding how to handle them.

In [12]:
# ── 3a. Payment type distribution ────────────────────────────
print("=== Payment type distribution ===")
print(df['payment_type'].value_counts())
print(df['payment_type'].value_counts(normalize=True).round(4) * 100)

# Voucher value distribution
print("\n=== Voucher payment_value distribution ===")
print(df[df['payment_type'] == 'voucher']['payment_value'].describe())

# Full details of not_defined rows
print("\n=== not_defined rows ===")
print(df[df['payment_type'] == 'not_defined'])

=== Payment type distribution ===
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
payment_type
credit_card    73.92
boleto         19.04
voucher         5.56
debit_card      1.47
not_defined     0.00
Name: proportion, dtype: float64

=== Voucher payment_value distribution ===
count    5775.000000
mean       65.703354
std       115.519185
min         0.000000
25%        18.035000
50%        39.280000
75%        80.000000
max      3184.340000
Name: payment_value, dtype: float64

=== not_defined rows ===
                               order_id  payment_sequential payment_type  \
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined   

       payment_installments  payment_value  
51280                     1            0.0  
57

In [13]:
# ── 3b. Check zero installments and zero payment value ────────
print("=== Zero installments by payment_type ===")
print(df[df['payment_installments'] == 0]['payment_type'].value_counts())

print("\n=== Zero payment_value by payment_type ===")
print(df[df['payment_value'] == 0]['payment_type'].value_counts())

=== Zero installments by payment_type ===
payment_type
credit_card    2
Name: count, dtype: int64

=== Zero payment_value by payment_type ===
payment_type
voucher        6
not_defined    3
Name: count, dtype: int64


In [14]:
# ── 3c. Investigate voucher rows with zero payment value ──────
# Check if these orders have other valid payment rows
# If yes, the zero-value voucher is an invalid券 attached to an otherwise valid order
print(f"Voucher with payment_value = 0: {len(df[(df['payment_type'] == 'voucher') & (df['payment_value'] == 0)])}")

voucher_zero_orders = df[(df['payment_type'] == 'voucher') & (df['payment_value'] == 0)]['order_id'].tolist()

print("\n=== All payment rows for orders with zero-value vouchers ===")
print(df[df['order_id'].isin(voucher_zero_orders)])

Voucher with payment_value = 0: 6

=== All payment rows for orders with zero-value vouchers ===
                                order_id  payment_sequential payment_type  \
4885    fa65dad1b0e818e3ccc5cb0e39231352                  27      voucher   
5163    8bcbe01d44d147f901cd3192671144db                   3      voucher   
9985    fa65dad1b0e818e3ccc5cb0e39231352                   4      voucher   
11755   45ed6e85398a87c253db47c2d9f48216                   2      voucher   
14321   fa65dad1b0e818e3ccc5cb0e39231352                   1      voucher   
17274   fa65dad1b0e818e3ccc5cb0e39231352                   9      voucher   
19565   fa65dad1b0e818e3ccc5cb0e39231352                  10      voucher   
19922   8bcbe01d44d147f901cd3192671144db                   4      voucher   
20963   8bcbe01d44d147f901cd3192671144db                   1  credit_card   
23074   fa65dad1b0e818e3ccc5cb0e39231352                   2      voucher   
24879   fa65dad1b0e818e3ccc5cb0e39231352                 

## Step 4 — Remove Invalid Rows

Three categories of invalid rows identified:
- `credit_card` with `payment_installments = 0`: logically invalid (2 rows)
- `voucher` with `payment_value = 0`: invalid voucher records attached to otherwise valid orders (6 rows)
- `not_defined` payment type with `payment_value = 0`: unclassified and worthless records (3 rows)

In [15]:
# ── 4. Remove invalid rows ────────────────────────────────────
rows_before = len(df)

mask1 = (df['payment_type'] == 'credit_card') & (df['payment_installments'] == 0)
mask2 = (df['payment_type'] == 'voucher') & (df['payment_value'] == 0)
mask3 = (df['payment_type'] == 'not_defined')

df = df[~(mask1 | mask2 | mask3)]
rows_after = len(df)

print(f"Rows removed: {rows_before - rows_after}")
print(f"Rows remaining: {rows_after}")

Rows removed: 11
Rows remaining: 103875


## Step 5 — Check Composite Primary Key Duplicates

No single column uniquely identifies a row in this table.  
The primary key is the combination of `order_id` + `payment_sequential`.  
For example, an order paying with both credit card and voucher would have rows: (order_A, 1), (order_A, 2).  
We verify this combination has no duplicates.

In [16]:
# ── 5. Check composite primary key duplicates ─────────────────
duplicate_count = df.duplicated(subset=['order_id', 'payment_sequential']).sum()
print(f"Duplicate (order_id + payment_sequential): {duplicate_count}")

Duplicate (order_id + payment_sequential): 0


## Step 6 — Export Cleaned Data

In [17]:
# ── 6. Export cleaned data ────────────────────────────────────
df.to_csv('../data/cleaned/order_payments_cleaned.csv', index=False)

print(f"Exported: {len(df)} rows")
print("Saved to: data/cleaned/order_payments_cleaned.csv")

Exported: 103875 rows
Saved to: data/cleaned/order_payments_cleaned.csv
